# Human-in-the-Loop (인간 개입 패턴)

**Human-in-the-Loop (HITL)** 패턴은 에이전트가 중요한 작업을 수행하기 전에
인간의 검토와 승인을 받는 패턴입니다.

이 패턴은:
- 보안이 중요한 작업 (결제, 데이터 삭제 등)
- 정확성이 중요한 작업 (법률 문서 검토, 의료 진단 등)
- 규정 준수가 필요한 작업 (규정 위반 가능성 있는 작업)

**참고**: [LangChain 공식 문서 - Human-in-the-Loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop)

In [ ]:
# LangSmith 추적 (선택적)

## 1. 기본 Human-in-the-Loop 설정

특정 도구 호출 시 인간의 승인을 기다리도록 설정합니다.

In [ ]:
# 결제 승인 요청 도구
def request_payment_approval(amount: int, purpose: str, requester: str) -> str:
# Human-in-the-Loop 미들웨어 정의
# 문서: 결정 타입은 approve, edit, reject 세 가지만 지원
        # 이 도구가 호출되면 인터럽트 발생
            # approve(승인), edit(수정 후 실행), reject(거부)
            # 도구 호출 시 설명 문구
    # 전역 prefix
# 에이전트 생성

## 2. 결제 승인 시나리오

결제 요청 → 인터럽트 → 관리자 승인/거부 → 워크플로우 재개

In [ ]:
# 결제 요청 정보
# 스레드 ID 생성
# 결제 승인 요청 → 인터럽트 발생
# 인터럽트 확인

### 2.1 관리자 승인

In [ ]:
# 관리자 결정: 승인
# HITL 재개용 Command
# 워크플로우 재개
# 승인 후 후속 처리

### 2.2 관리자 거부

In [ ]:
# 새로운 스레드로 거부 시나리오 테스트
# 결제 요청
# 관리자 거부 결정
# 거부 후 후속 처리

### 2.3 관리자 수정(edit)

`edit` 결정으로 도구 인자를 수정한 뒤 실행할 수 있습니다.
문서: https://docs.langchain.com/oss/python/langchain/human-in-the-loop

In [ ]:
    # edit: 금액을 800_000으로 수정한 뒤 실행

## 3. 다중 도구 인터럽트

여러 도구에 대해 인터럽트를 설정할 수 있습니다.

In [ ]:
# 여러 도구 정의
def delete_user_data(user_id: str) -> str:
def send_bulk_email(recipients: list[str], subject: str) -> str:
def update_database(query: str) -> str:
# 다중 도구 HITL 미들웨어
# 다중 도구 에이전트
# system_prompt: 사용자 요청 시 반드시 해당 도구를 호출하도록 유도 → 인터럽트가 실제로 걸리게 함

## 4. 스트리밍과 인터럽트

스트리밍 모드에서 인터럽트를 처리하는 방법입니다.

In [ ]:
# 스트리밍용 스레드 ID 생성
# ------------------------------------------------------------
# 에이전트 실행 중 발생하는 상태 이벤트와 모델 응답을 동시에 실시간 처리하는 스트리밍 루프
# ------------------------------------------------------------
# stream_mode:
#   - "updates"   → 상태 변화, interrupt, tool call 등 이벤트 수신
#   - "messages"  → 모델이 생성하는 토큰 스트리밍 수신
#
# 반환값은 (mode, chunk) 형태의 generator
# SystemMessage로 이번 턴에서 delete_user_data 도구 호출을 명시 → 인터럽트 발생 보장
        # interrupt 이벤트가 발생한 경우
            # 여러 interrupt가 동시에 발생할 수 있음
                # interrupt 객체 또는 dict 처리
                # 대기 중인 action 요청 출력
    # 모델 토큰 스트리밍 처리
        # chunk는 (token, metadata) 형태
        # token.content가 존재하면 즉시 출력
        # flush=True → 실시간 출력

## 주요 포인트 정리

1. **인터럽트 설정**: `interrupt_on`으로 특정 도구 호출 시 인터럽트 발생
2. **결정 옵션**: `allowed_decisions`는 문서 기준 `approve`, `edit`, `reject` 세 가지만 지원
3. **체크포인터 필수**: HITL은 반드시 checkpointer 필요
4. **Command로 재개**: `Command(resume={"decisions": [...]})` 로 워크플로우 재개
5. **edit**: 수정 후 실행 시 `edited_action`에 `name`, `args` 전달
6. **스트리밍**: `stream_mode=["updates", "messages"]` 로 진행 상황과 토큰 스트리밍

- [streamlit-llm_LangChain/310_Personal_Assistant_App.py](streamlit-llm_LangChain/350_Human_In_The_Loop_App.py)에서 웹 UI 구현